# Utah FORGE 01 Preprocess And Segment

Apply dataset-specific preprocessing, construct the physical state, segment events, and export a selected cycle.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import get_dataset_config
from src.datasets.utah_forge import build_utah_forge_summary
from src.io.utah_forge import load_utah_forge_dataset
from src.preprocess.utah_forge import build_utah_forge_state
from src.segmentation.utah_forge import segment_utah_forge_events
from src.segmentation.common import summarize_segments, extract_segment
from src.utils.plotting import plot_signal_panel, plot_segment_boundaries

CONFIG = get_dataset_config("utah_forge")
RESULTS_DIR = CONFIG.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
summary = build_utah_forge_summary()
if not summary.get("analysis_ready", False):
    raise RuntimeError(
        "This dataset is not ready for preprocessing/segmentation. "
        f"Check {RESULTS_DIR / 'dataset_summary.json'} and add the required raw file(s) under {CONFIG.raw_dir}."
    )

raw_df, load_summary = load_utah_forge_dataset()
state_df, state_metadata = build_utah_forge_state(raw_df, load_summary["column_mapping"])
segments = segment_utah_forge_events(state_df, tau_col="tau", min_cycle_length=CONFIG.segmentation["min_cycle_length"])
if not segments:
    segments = [(0, len(state_df))]
selected_cycle = extract_segment(state_df, *segments[0])

cycle_path = RESULTS_DIR / "selected_cycle_001.csv"
cycle_meta_path = RESULTS_DIR / "selected_cycle_001_metadata.json"
seg_summary_path = RESULTS_DIR / "segmentation_summary.json"
selected_cycle.to_csv(cycle_path, index=False)
cycle_metadata = {
    "dataset": "utah_forge",
    "state_columns": ["tau", "V"],
    "state_labels": ["tau", "V"],
    "velocity_mode": state_metadata.get("velocity_mode", CONFIG.velocity_mode),
    "cycle_path": str(cycle_path),
    "selected_segment": [int(segments[0][0]), int(segments[0][1])],
}
cycle_meta_path.write_text(json.dumps(cycle_metadata, indent=2), encoding="utf-8")
seg_summary_path.write_text(
    json.dumps(
        {
            "dataset": "utah_forge",
            "segment_summary": summarize_segments(segments),
            "segments": segments,
            "selected_cycle_path": str(cycle_path),
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Selected cycle saved to:", cycle_path)
print("Segment summary:", summarize_segments(segments))


Selected cycle saved to: C:\Users\carla\Desktop\EECE 798K\Project\results\utah_forge\selected_cycle_001.csv
Segment summary: {'count': 14, 'lengths': [30563, 504645, 164202, 48742, 22629, 3948, 128373, 26703, 54283, 489318, 29357, 184231, 16193, 11137], 'mean_length': 122451.71428571429, 'min_length': 3948, 'max_length': 504645}


In [3]:
plot_signal_panel(state_df.head(min(4000, len(state_df))), "time", ["tau", "V"], "Utah FORGE physical state preview")
plot_segment_boundaries(state_df, "time", "tau", segments, "Utah FORGE segmentation boundaries")
